# Retention Frontier Target Revision Readiness v1

This notebook is the executable evidence for the current model-selection handoff. It verifies why no actor should be retained yet, why SOXLBacktest remains a qlib buy/sell reference baseline, and why the next model-training loop should start with a fold-local wf_2020 bull/reentry target revision before any retained actor training.

## Governance

- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `model_retention_frontier_v1`.
- Role: `diagnose`.
- Target window: required walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024` plus YTD alignment through the dashboard replay safe end.
- Changed layer: `model_selection` / `selection`.
- External reference: `soxlbacktest_qlib_buysell_reference` as a buy/sell baseline, not a runnable actor.
- Hypothesis: the frontier should advance from completed metric handoff to the failed-fold objective audit when target revision is the next actionable step.
- Acceptance: strict retention remains blocked, the frontier selected source is `target_revision_ready`, and the next action is the fold-local bull/reentry target rebuild.

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'runtime').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

ROOT = find_repo_root(Path.cwd())
OUT = ROOT / 'runtime' / 'model_selection' / 'card' / 'retention_frontier_target_revision_readiness_v1'
OUT.mkdir(parents=True, exist_ok=True)

def load_json(rel):
    return json.loads((ROOT / rel).read_text(encoding='utf-8'))

frontier = load_json('runtime/model_selection/card/model_retention_frontier_v1/summary.json')
gate = load_json('runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json')
metric_handoff = load_json('runtime/model_selection/card/cap10_metric_replay_handoff_audit_v1/summary.json')
failed_fold = load_json('runtime/model_selection/card/cap10_wf2020_failed_fold_objective_audit_v1/summary.json')
freshness = load_json('dashboard_data/model_replay_freshness.json')
manifest = load_json('dashboard_data/v2/manifest.json')

frontier_rows = pd.DataFrame(frontier['rows'])
top_rows = frontier_rows.head(10).copy()
top_rows.to_csv(OUT / 'frontier_top_rows.csv', index=False)

selected = frontier['rows'][0]
selected

{'decision': 'wf2020_failed_fold_requires_bull_reentry_target_revision_no_promotion',
 'experiment': 'cap10_wf2020_failed_fold_objective_audit_v1',
 'failed_wf_windows': '',
 'frontier_rank': 1,
 'handoff_completed_by': '',
 'min_wf_relative_return_pp': None,
 'missing_wf_windows': 'wf_2020,wf_2022,wf_2023_2024',
 'next_action': 'build_fold_local_bull_reentry_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay',
 'primary_blocker': 'target_revision_ready',
 'promotion_allowed': False,
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
 'relative_return_gap_to_target_pp': None,
 'source_path': 'runtime/model_selection/card/cap10_wf2020_failed_fold_objective_audit_v1/summary.json',
 'target_revision_ready': True,
 'wf_drawdown_gap_pct': None,
 'wf_windows_present': '',
 'worst_wf_max_drawdown_pct': None,
 'ytd_gap_to_best_pct': None,
 'ytd_gap_to_soxlbacktest_pct': None,
 'ytd_return_pct': None}

In [2]:
gate_summary = gate['summary']
fresh_counts = freshness.get('counts', {})
auto_update = manifest.get('auto_update', {})
loaded_tasks = [task for task in auto_update.get('tasks', []) if task.get('loaded') is True]
metric_selected = metric_handoff['selected_by_fold']['wf_2020']
failed_readout = failed_fold['wf_2020_selected']

readout = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'target_revision_before_retained_actor_training',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'recommended_next_source': frontier['summary']['recommended_next_source'],
    'recommended_primary_blocker': frontier['summary']['recommended_primary_blocker'],
    'recommended_next_action': frontier['summary']['recommended_next_action'],
    'strict_gate_passing_profile_count': gate_summary['passing_profile_count'],
    'registry_keep_actor_strict_violation_count': gate_summary['registry_keep_actor_strict_violation_count'],
    'registry_keep_actor_protected_exception_count': gate_summary['registry_keep_actor_protected_exception_count'],
    'soxlbacktest_reference_return_pct': gate_summary['reference_return_pct'],
    'soxlbacktest_reference_max_drawdown_pct': gate_summary['reference_max_drawdown_pct'],
    'required_best_return_pct': gate_summary['required_best_return_pct'],
    'max_walkforward_drawdown_pct': gate_summary['max_walkforward_drawdown_pct'],
    'required_wf_windows': gate_summary['required_wf_windows'],
    'dashboard_replay_ready_count': fresh_counts.get('current', 0) + fresh_counts.get('refreshed', 0),
    'dashboard_replay_profile_count': len(freshness.get('profiles', [])),
    'dashboard_replay_skipped_count': fresh_counts.get('skipped', 0),
    'safe_backtest_end': freshness.get('safe_backtest_end'),
    'auto_update_configured': auto_update.get('configured'),
    'auto_update_loaded_task_count': len(loaded_tasks),
    'manual_refresh_command': auto_update.get('manual_refresh_command'),
    'wf2020_selected_candidate': metric_selected['selected_candidate'],
    'wf2020_selected_max_dd_pct': metric_selected['selected_max_dd'],
    'wf2020_selected_relative_return_pp': metric_selected['selected_relative_return'],
    'wf2020_relative_gap_to_target_pp': failed_readout['relative_gap_to_target_pp'],
    'wf2020_maxdd_gap_to_target_pp': failed_readout['maxdd_gap_to_target_pp'],
    'source_paths': [
        'runtime/model_selection/card/model_retention_frontier_v1/summary.json',
        'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json',
        'runtime/model_selection/card/cap10_metric_replay_handoff_audit_v1/summary.json',
        'runtime/model_selection/card/cap10_wf2020_failed_fold_objective_audit_v1/summary.json',
        'dashboard_data/model_replay_freshness.json',
        'dashboard_data/v2/manifest.json',
    ],
}
(OUT / 'summary.json').write_text(json.dumps(readout, indent=2, sort_keys=True) + '\n', encoding='utf-8')
pd.DataFrame([readout]).to_csv(OUT / 'target_revision_readout.csv', index=False)
readout

{'generated_at': '2026-06-13T23:03:43.078860+00:00',
 'decision': 'target_revision_before_retained_actor_training',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'recommended_next_source': 'cap10_wf2020_failed_fold_objective_audit_v1',
 'recommended_primary_blocker': 'target_revision_ready',
 'recommended_next_action': 'build_fold_local_bull_reentry_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay',
 'strict_gate_passing_profile_count': 0,
 'registry_keep_actor_strict_violation_count': 7,
 'registry_keep_actor_protected_exception_count': 7,
 'soxlbacktest_reference_return_pct': 132.077978,
 'soxlbacktest_reference_max_drawdown_pct': 13.554089,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'dashboard_replay_ready_count': 0,
 'dashboard_replay_profile_count': 7,
 'dashboard_replay_skipped_count': 7,
 'safe_backtest_end': '2026-06-09',
 'auto_update_c

In [3]:
display_columns = [
    'frontier_rank',
    'experiment',
    'primary_blocker',
    'next_action',
    'target_revision_ready',
    'failed_wf_windows',
    'worst_wf_max_drawdown_pct',
    'relative_return_gap_to_target_pp',
]
top_rows[display_columns]

,frontier_rank,experiment,primary_blocker,next_action,target_revision_ready,failed_wf_windows,worst_wf_max_drawdown_pct,relative_return_gap_to_target_pp
0,1,cap10_wf2020_failed_fold_objective_audit_v1,target_revision_ready,build_fold_local_bull_reentry_participation_ta...,True,,NaN,NaN
1,2,cap10_metric_replay_handoff_audit_v1,walkforward_relative_return_gap,inspect_failed_fold_drawdown_or_reward_drift_b...,False,wf_2020,18.8,322.75
2,3,cap10_wf2020_participation_failure_attribution_v1,missing_walkforward_evidence,rebuild_wf2020_bull_participation_target_befor...,False,,NaN,NaN
3,4,conservative_rebound_release_gate_v1,missing_walkforward_evidence,revise_target_semantics_with_sparse_soxx_sprea...,False,,NaN,NaN
4,5,constrained_checkpoint_selector_v1,missing_walkforward_evidence,generate_reward_constrained_short_training_can...,False,,NaN,NaN
5,6,fold_local_rebound_priority_conflict_v1,missing_walkforward_evidence,revise_state_objective_target_semantics_before...,False,,NaN,NaN
6,7,official_replay_bad_interval_feature_audit_v1,missing_walkforward_evidence,build official-replay-gated risk-state checkpo...,False,,NaN,NaN
7,8,official_replay_candidate_mechanism_pk_v1,missing_walkforward_evidence,stress-test short-volshock action-cap overlay ...,False,,NaN,NaN
8,9,quantbench_ensemble_current_walkforward_v1,missing_walkforward_evidence,use_failure_columns_to_tune_bull_release_or_dr...,False,,NaN,NaN
9,10,short_volshock_action_cap_stress_test_v1,missing_walkforward_evidence,reward-timing objective should target at least...,False,,NaN,NaN


## Conclusion

- What ran: refreshed retention gate/frontier evidence and inspected the current top model-selection handoff.
- Key metrics: strict retention still has `0` passing replayable profiles; `7` keep-actor rows are protected exceptions, not promotable models. SOXLBacktest qlib reference is `132.08%` YTD return with `13.55%` MaxDD, while the current best-alignment floor is `213.14%` and every retained model must pass `wf_2020`, `wf_2022`, and `wf_2023_2024` with MaxDD `<20%`.
- Failure diagnosis: `cap10_metric_replay_handoff_audit_v1` already completed the replay handoff; the remaining failed fold is `wf_2020`, where the selected candidate has MaxDD `14.98%` but relative return `-292.75pp`, a `322.75pp` shortfall to the `+30pp` target. This is bull/reentry underparticipation, not a drawdown breach.
- Decision: do not retain actors and do not run full retained training from the current target as-is. The next actionable source is `cap10_wf2020_failed_fold_objective_audit_v1` with blocker `target_revision_ready`.
- Next action: build a fold-local bull/reentry participation target, then rerun the three-fold BC/checkpoint metric replay against the qlib SOXLBacktest buy/sell reference. Dashboard replay remains non-runnable for the old protected actors until a real retained actor exists and refreshes cleanly.